# CNN Image Classifier — CIFAR-10

Classifies images into 10 categories using a custom convolutional neural 
network built from scratch, achieving 83.4% validation accuracy.

**Approach:** Built a 3-block CNN architecture (Conv2D + BatchNorm + 
MaxPooling + Dropout) trained from scratch on 50,000 32x32 images across 
10 classes. Evaluated via training curves, confusion matrix, and 
per-class precision/recall.

**Result:** 83.4% validation accuracy. Strongest performance on automobile 
(93% F1) and ship (90% F1) classes; weakest on cat (70% F1) — a known 
challenge in this dataset due to pose and lighting variation, consistent 
with published CIFAR-10 benchmarks.

# Imports & Load Dataset

In [1]:
# Imports & Load Dataset
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pickle

def unpickle(file):
    with open(file, 'rb') as fo:
        d = pickle.load(fo, encoding='bytes')
    return d

base_path = '/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/'

# Load all 5 training batches
X_train_list, y_train_list = [], []
for i in range(1, 6):
    batch = unpickle(base_path + f'data_batch_{i}')
    X_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])

X_train = np.concatenate(X_train_list).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_train = np.concatenate(y_train_list)

# Load test batch
test_batch = unpickle(base_path + 'test_batch')
X_test = test_batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_test = np.array(test_batch[b'labels'])

# Load class names
meta = unpickle(base_path + 'batches.meta')
class_names = [name.decode('utf-8') for name in meta[b'label_names']]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Classes: {class_names}")

2026-06-20 19:07:57.350848: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781982477.570185      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781982477.637579      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781982478.162463      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781982478.162501      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781982478.162503      58 computation_placer.cc:177] computation placer alr

Train shape: (50000, 32, 32, 3), Test shape: (10000, 32, 32, 3)
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# Preprocessing & Sample Visualization

In [ ]:
# Preprocessing & Sample Visualization
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(class_names[y_train[i]], fontsize=11)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

# Build CNN Architecture

In [ ]:
# Build CNN Architecture
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Train the Model

In [ ]:
# Train the Model
history = model.fit(
    X_train_norm, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_norm, y_test),
    verbose=1
)

# Training Curves

In [ ]:
# Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='#3498db')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='#e74c3c')
axes[0].set_title('Model Accuracy Over Training', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss', color='#3498db')
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='#e74c3c')
axes[1].set_title('Model Loss Over Training', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion Matrix

In [ ]:
# Confusion Matrix
y_pred_proba = model.predict(X_test_norm)
y_pred = np.argmax(y_pred_proba, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — CNN Image Classifier', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

# Sample Predictions — Correct vs Incorrect

In [ ]:
# Sample Predictions — Correct vs Incorrect
correct_idx = np.where(y_pred == y_test)[0]
incorrect_idx = np.where(y_pred != y_test)[0]

fig, axes = plt.subplots(2, 5, figsize=(15, 7))

# Top row: correct predictions
for i, idx in enumerate(np.random.choice(correct_idx, 5, replace=False)):
    axes[0, i].imshow(X_test[idx])
    axes[0, i].set_title(f"✓ {class_names[y_pred[idx]]}", color='green', fontsize=11)
    axes[0, i].axis('off')

# Bottom row: incorrect predictions
for i, idx in enumerate(np.random.choice(incorrect_idx, 5, replace=False)):
    axes[1, i].imshow(X_test[idx])
    axes[1, i].set_title(f"✗ {class_names[y_pred[idx]]}\n(actual: {class_names[y_test[idx]]})", 
                          color='red', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Sample Predictions — Correct (top) vs Incorrect (bottom)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()